In [1]:
# ============================================================
# FUNCTION 7 — WEEK 6 CLEAN, MEMORY-SAFE REBUILD
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 7 data
# ------------------------------------------------------------

X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–5 exactly once
# ------------------------------------------------------------

weekly_X = np.array([
    [0.148437, 0.535927, 0.276397, 0.096809, 0.345999, 0.751013],
    [0.033169, 0.329738, 0.366358, 0.234301, 0.286315, 0.704927],
    [0.000000, 0.300093, 0.304819, 0.158057, 0.252468, 0.776583],
    [0.000000, 0.321413, 0.400365, 0.258320, 0.285038, 0.762744],
    [0.000000, 0.338035, 0.346669, 0.250368, 0.283186, 0.625396]
])

weekly_Y = np.array([
    1.1745584410367518,
    2.4608579737515917,
    1.8105475948256518,
    2.287474907540469,
    2.3365189225488012
])

X = np.vstack([X, weekly_X])
Y = np.concatenate([Y, weekly_Y])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Week 6 X shape:", X.shape)
print("Week 6 Y shape:", Y.shape)

print("\nBest observed input:")
print(best_x)

print("Best observed output:")
print(best_y)

print("\nWeek 5 input:")
print(weekly_X[-1])

print("Week 5 output:")
print(weekly_Y[-1])

Week 6 X shape: (35, 6)
Week 6 Y shape: (35,)

Best observed input:
[0.033169 0.329738 0.366358 0.234301 0.286315 0.704927]
Best observed output:
2.4608579737515917

Week 5 input:
[0.       0.338035 0.346669 0.250368 0.283186 0.625396]
Week 5 output:
2.3365189225488012


In [2]:
# ============================================================
# 3. FIT ARD GAUSSIAN PROCESS
# ============================================================

kernel = (
    C(1.0, (1e-3, 1e3))
    *
    Matern(
        length_scale=np.full(6, 0.30),
        length_scale_bounds=(0.02, 3.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-10, 1e-2)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=7,
    random_state=67
)

gp.fit(X, Y)

print("Fitted kernel:")
print(gp.kernel_)

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nFitted lengthscales:")
print(lengthscales)

print("\nNormalised input influence:")

for i, value in enumerate(importance, start=1):
    print(f"x{i}: {value:.4f}")

Fitted kernel:
0.643**2 * Matern(length_scale=[0.339, 3, 3, 0.131, 0.142, 3], nu=2.5) + WhiteKernel(noise_level=1.47e-10)

Fitted lengthscales:
[0.33852575 3.         3.         0.13138254 0.14185055 3.        ]

Normalised input influence:
x1: 0.1587
x2: 0.0179
x3: 0.0179
x4: 0.4089
x5: 0.3787
x6: 0.0179


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 3.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [3]:
# ============================================================
# 4. GENERATE WEEK 6 CANDIDATES
# ============================================================

rng = np.random.default_rng(67)

week2_point = np.array([
    0.033169,
    0.329738,
    0.366358,
    0.234301,
    0.286315,
    0.704927
])

week5_point = np.array([
    0.000000,
    0.338035,
    0.346669,
    0.250368,
    0.283186,
    0.625396
])

week4_point = np.array([
    0.000000,
    0.321413,
    0.400365,
    0.258320,
    0.285038,
    0.762744
])

# ------------------------------------------------------------
# A. Very local around Week 2 best
# ------------------------------------------------------------

very_local = rng.normal(
    loc=week2_point,
    scale=[
        0.006,
        0.010,
        0.010,
        0.010,
        0.008,
        0.012
    ],
    size=(7_000, 6)
)

# ------------------------------------------------------------
# B. Local around Week 2
# ------------------------------------------------------------

local_best = rng.normal(
    loc=week2_point,
    scale=[
        0.018,
        0.025,
        0.025,
        0.025,
        0.020,
        0.030
    ],
    size=(8_000, 6)
)

# ------------------------------------------------------------
# C. Local around Week 5
# ------------------------------------------------------------

local_week5 = rng.normal(
    loc=week5_point,
    scale=[
        0.010,
        0.020,
        0.020,
        0.020,
        0.015,
        0.030
    ],
    size=(6_000, 6)
)

# ------------------------------------------------------------
# D. Blend Week 2 and Week 5
# ------------------------------------------------------------

weights = rng.uniform(
    0,
    1,
    size=(5_000, 1)
)

blend_25 = (
    weights * week2_point
    +
    (1 - weights) * week5_point
)

blend_25 += rng.normal(
    0,
    [
        0.008,
        0.012,
        0.012,
        0.012,
        0.010,
        0.018
    ],
    size=(5_000, 6)
)

# ------------------------------------------------------------
# E. Small blend across Week 2, Week 4 and Week 5 region
# ------------------------------------------------------------

weights_24 = rng.uniform(
    0,
    1,
    size=(3_000, 1)
)

blend_24 = (
    weights_24 * week2_point
    +
    (1 - weights_24) * week4_point
)

blend_24 += rng.normal(
    0,
    [
        0.008,
        0.015,
        0.015,
        0.015,
        0.012,
        0.020
    ],
    size=(3_000, 6)
)

candidates = np.vstack([
    very_local,
    local_best,
    local_week5,
    blend_25,
    blend_24
])

candidates = np.clip(candidates, 0, 1)

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 29000


In [4]:
# ============================================================
# 5. REMOVE NEAR-DUPLICATES
# ============================================================

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

keep = minimum_distance > 0.0035

candidates = candidates[keep]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 28999


In [5]:
# ============================================================
# 6. PREDICT IN BATCHES
# ============================================================

def predict_in_batches(
    model,
    points,
    batch_size=2000
):

    means = []
    standard_deviations = []

    for start in range(
        0,
        len(points),
        batch_size
    ):

        batch = points[
            start:start + batch_size
        ]

        batch_mean, batch_std = model.predict(
            batch,
            return_std=True
        )

        means.append(batch_mean)
        standard_deviations.append(batch_std)

    return (
        np.concatenate(means),
        np.concatenate(standard_deviations)
    )

mean, std = predict_in_batches(
    gp,
    candidates,
    batch_size=2000
)

In [6]:
# ============================================================
# 7. EXPECTED IMPROVEMENT
# ============================================================

xi = 0.003

improvement = (
    mean
    - best_y
    - xi
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        +
        std * norm.pdf(z)
    )

ei[std < 1e-12] = 0


# ------------------------------------------------------------
# Small UCB component
# ------------------------------------------------------------

kappa = 0.35

ucb = (
    mean
    + kappa * std
)

In [7]:
# ============================================================
# 8. FILTER WEAK PREDICTIONS
# ============================================================

mean_filter = (
    mean >= best_y - 0.15
)

if mean_filter.sum() < 100:

    mean_filter = (
        mean >= np.percentile(
            mean,
            90
        )
    )

filtered_candidates = candidates[
    mean_filter
]

filtered_mean = mean[
    mean_filter
]

filtered_std = std[
    mean_filter
]

filtered_ei = ei[
    mean_filter
]

filtered_ucb = ucb[
    mean_filter
]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

Candidates passing mean filter: 22129


In [8]:
# ============================================================
# 9. ACQUISITION SCORE
# ============================================================

ei_norm = (
    filtered_ei
    - filtered_ei.min()
) / (
    np.ptp(filtered_ei)
    + 1e-12
)

ucb_norm = (
    filtered_ucb
    - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb)
    + 1e-12
)

# Week 2 is still best, so favour exploitation
acquisition = (
    0.88 * ei_norm
    +
    0.12 * ucb_norm
)

chosen_index = np.argmax(
    acquisition
)

week6_query = filtered_candidates[
    chosen_index
]

In [9]:
# ============================================================
# 10. FUNCTION 7 — WEEK 6 PORTAL OUTPUT
# ============================================================

print("\nSuggested Week 6 query:")
print(week6_query)

print("\nPortal format:")
print(
    "-".join(
        f"{value:.6f}"
        for value in week6_query
    )
)

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted uncertainty:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "UCB:",
    filtered_ucb[chosen_index]
)

print(
    "Distance from Week 2 best:",
    np.linalg.norm(
        week6_query
        - week2_point
    )
)

print(
    "Distance from Week 5:",
    np.linalg.norm(
        week6_query
        - week5_point
    )
)

print(
    "Distance from Week 4:",
    np.linalg.norm(
        week6_query
        - week4_point
    )
)


Suggested Week 6 query:
[0.09368401 0.32560422 0.37137904 0.23046737 0.28145994 0.6388949 ]

Portal format:
0.093684-0.325604-0.371379-0.230467-0.281460-0.638895

Predicted mean: 2.4524667423697375
Predicted uncertainty: 0.08171676741783315
Expected Improvement: 0.027220891270026217
UCB: 2.481067610965979
Distance from Week 2 best: 0.09001595068744839
Distance from Week 5: 0.10061330984062242
Distance from Week 4: 0.16050425532264567
